# Caso C · 03 Features para detección de anomalías HVAC

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir features que separen normal de fallo: ΔT, duty cycles, ratios fan/valve, lags.


## 2. Qué se aprende

- ΔT como indicador de transferencia de calor.
- Duty cycle del HVAC (% on en una ventana).
- Ratio anomalía cuando válvula activa pero fan apagado.
- Por qué los autoencoders prefieren features escaladas.


## 3. Contexto del caso de uso

Las features físicamente significativas mejoran la interpretabilidad y reducen el espacio de búsqueda.


## 4. Relación con CENTINELA+

Mismas features se calcularían sobre `simarro-prod` cuando llegue un ticket de incidencia — para reproducir la firma.


## 5. Relación con Medallion

Lee plata, escribe oro local.


## 6. Datos de entrada

Plata mock (CSV) + etiquetas.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica para oro (parquet).


## 9. Carga de datos o mock

Cargamos plata y etiquetas.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "lbnl_fdd_rtu_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).sort_values("timestamp").set_index("timestamp")
df.head()


,OA_TEMP,SA_TEMP,RA_TEMP,CCV,FAN_STATE,OCCU_MOD,is_fault,fault_type
timestamp,,,,,,,,
2024-06-01 00:00:00+00:00,22.15,16.0,22.98,0,0,0,0,normal
2024-06-01 00:01:00+00:00,21.51,16.0,22.97,0,0,0,0,normal
2024-06-01 00:02:00+00:00,22.45,16.0,23.17,0,0,0,0,normal
2024-06-01 00:03:00+00:00,22.57,16.0,22.30,0,0,0,0,normal
2024-06-01 00:04:00+00:00,21.16,16.0,23.04,0,0,0,0,normal


## 10. Exploración paso a paso

Computamos features y discutimos cobertura.


In [3]:
def make_features(d):
    f = pd.DataFrame(index=d.index)
    f["dt_supply_return"] = d["RA_TEMP"] - d["SA_TEMP"]
    f["dt_supply_outdoor"] = d["OA_TEMP"] - d["SA_TEMP"]
    f["valve"] = d["CCV"]
    f["fan"] = d["FAN_STATE"]
    f["fan_valve_diff"] = f["valve"] - f["fan"]
    f["valve_duty_60"] = f["valve"].rolling("60min").mean()
    f["fan_duty_60"] = f["fan"].rolling("60min").mean()
    f["dt_lag_15"] = f["dt_supply_return"].shift(15)
    f["dt_change_15"] = f["dt_supply_return"] - f["dt_lag_15"]
    f["is_fault"] = d["is_fault"]
    f["fault_type"] = d["fault_type"]
    return f.dropna()

X = make_features(df)
X.head()


,dt_supply_return,dt_supply_outdoor,valve,fan,fan_valve_diff,valve_duty_60,fan_duty_60,dt_lag_15,dt_change_15,is_fault,fault_type
timestamp,,,,,,,,,,,
2024-06-01 00:15:00+00:00,6.84,6.09,0,0,0,0.0,0.0,6.98,-0.14,0,normal
2024-06-01 00:16:00+00:00,6.77,6.74,0,0,0,0.0,0.0,6.97,-0.20,0,normal
2024-06-01 00:17:00+00:00,7.01,6.11,0,0,0,0.0,0.0,7.17,-0.16,0,normal
2024-06-01 00:18:00+00:00,6.83,7.07,0,0,0,0.0,0.0,6.30,0.53,0,normal
2024-06-01 00:19:00+00:00,6.57,6.64,0,0,0,0.0,0.0,7.04,-0.47,0,normal


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Persistimos.


In [4]:
out_dir = ROOT / "output" / "case_C"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "hvac_features.parquet"
X.drop(columns=["fault_type"]).to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)} ({len(X)})")


Wrote output\case_C\hvac_features.parquet (20145)


## 13. Visualizaciones explicativas

Distribución de `dt_supply_return` separada por `is_fault`.


In [5]:
plot_distribution(X.assign(grupo=np.where(X.is_fault == 1, "fault", "normal")),
                  column="dt_supply_return", by="grupo", title="ΔT_return-supply normal vs fallo")


<Axes: title={'center': 'ΔT_return-supply normal vs fallo'}, xlabel='dt_supply_return', ylabel='frecuencia'>

## 14. Validaciones

El target debe estar balanceado lo suficiente.


In [6]:
counts = X["is_fault"].value_counts(normalize=True)
print(counts)
assert counts.min() > 0.02


is_fault
0    0.891288
1    0.108712
Name: proportion, dtype: float64


## 15. Errores comunes

1. **Olvidar shift en rolling**: leakage.
2. **Usar lag mayor que ventana**: NaN al inicio.
3. **Mezclar fallos en mismo modelo binario sin codificar tipo**.


## 16. Ejercicios propuestos

1. Añade `valve_duty_15` y compara feature importance.
2. Discute si SHAP funcionaría mejor con o sin escalado.
3. Construye `fault_id` único por episodio.


## 17. Cómo se reutiliza con datos reales

`make_features` es pura: misma función sobre cualquier `simarro-prod` data.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/04_isolation_forest_autoencoder.ipynb`.
- Documento web del caso: `docs/use-cases/case-c-hvac-anomaly.md`.


## 19. Marco teórico (nivel doctoral)

### Isolation Forest (Liu et al. 2008)

Score basado en la profundidad media $E[h(x)]$ del path al aislar $x$ en
árboles binarios construidos por particiones aleatorias:

$$
s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}
$$

con $c(n) = 2 H(n-1) - 2(n-1)/n$ y $H(i)$ harmonic number. Anomalía si
$s(x) \to 1$, normal si $s(x) \to 0.5$.

### Autoencoder (Hinton & Salakhutdinov 2006)

$$
\hat{x} = D(E(x)), \quad E: \mathbb{R}^d \to \mathbb{R}^k, \quad D: \mathbb{R}^k \to \mathbb{R}^d, \quad k \ll d
$$

con $k = 8$ neuronas en bottleneck para $d = 24$ features. Score:

$$
e(x) = \| x - \hat{x} \|_2^2, \quad \theta = \mu_e + 3\sigma_e
$$

### Catálogo de fallos (ADR-010)

| Tipo | Variable afectada | Signature |
|---|---|---|
| `sensor_drift` | `temperature_01` | drift lineal $+0.5$ °C/h |
| `valve_stuck` | `valve_state`, $T_{int}$ | $\Delta T \to 0$ tras setpoint change |
| `fan_failure` | `fan_speed_01_state`, $T_{supply}$ | $\dot V \to 0$, $T_{supply} \to T_{coil}$ |
| `refrigerant_low` | $T_{supply} - T_{return}$ | $\Delta T_{cool}$ cae 50 % |

### Métricas

$$
\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}, \quad
\text{TPR}@1\%\text{FPR} = \text{Recall} \mid \text{FPR} \leq 0.01
$$

Objetivos: $\text{F1} \geq 0.85$, $\text{TPR}@1\%\text{FPR} \geq 0.7$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Detectar fallos HVAC antes de que se manifiesten en quejas de alumnos o averías catastróficas tiene **doble valor**: ahorro operativo (mantenimiento predictivo en lugar de reactivo) y diferenciador comercial frente a competidores BMS sin IA.

### ROI estimado

| Concepto | Valor anual |
|---|---|
| Detección temprana avería catastrófica (~2/año × 3 500 €) | +7 000 € |
| Reducción downtime (2 h × 200 días) | +800 € |
| **Bruto** | **+7 800 €/año** |
| Coste integración | -2 000 € one-time |
| **Payback** | **~3 meses** |

### Riesgos y mitigaciones

- False positives → fatiga de alarmas. Tunear umbral con percentil 99.
- Drift en HVAC envejecido: incluir age-feature.


## 21. Bibliografía y referencias

- Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). *Isolation Forest*. ICDM '08.
- Hinton, G. & Salakhutdinov, R. (2006). *Reducing the Dimensionality of Data with Neural Networks*. Science 313(5786).
- Granderson, J. et al. (2020). *Building Fault Detection Data to Aid Diagnostic Algorithm Creation and Performance Testing*. Scientific Data 7.
- ASHRAE (2021). *Guideline 36-2021 — High-Performance Sequences of Operation for HVAC Systems*.
